# Extract dynamically downscaled projections

In [ ]:
import sys
import os
import glob
import xarray as xr
sys.path.append('/g/data/qz03/wp3/users/jb6465/code/bmkg')
from utils import *

In [ ]:
import dask
import tempfile
from dask.diagnostics import ProgressBar
from dask.distributed import Client, LocalCluster

dask.config.set({
    'array.chunk-size': "256 MiB",
    'array.slicing.split_large_chunks': True, #This can make AXIOM very slow
    'distributed.comm.timeouts.connect': '120s',
    'distributed.comm.timeouts.tcp': '120s',
    'distributed.comm.retry.count': 10,
    'distributed.scheduler.allowed-failures': 20,
    "distributed.scheduler.worker-saturation": 1.1, #This should use the new behaviour which helps with memory pile up
})

# client = dask.distributed.Client()
client = Client(n_workers=10, threads_per_worker=1, local_directory = tempfile.mkdtemp(), memory_limit = "63000mb")
client

In [ ]:
sys.path.append('/g/data/qz03/wp3/users/jb6465/code/git-repo/submodules/dataset_finder')
from dataset_finder import *

In [ ]:
timescale='mon'
ACS_downscaled_historical = get_datasets("ACS_DS", empat = "historical" , timescale=timescale, var='sfcWindmax', grid=("AUS-15", "AUS-10i"))#, "AUS-18")) #get_datasets("CORDEX-CMIP6_Regridded", scenario = "historical" , timescale='mon', grid="gr1.5")
ACS_downscaled_ssp370 = get_datasets("ACS_DS", empat = "ssp370" , timescale=timescale, var='sfcWindmax', grid=("AUS-15", "AUS-10i"))#, "AUS-18")) #get_datasets("CORDEX-CMIP6_Regridded", scenario = "ssp370" , timescale='mon', grid="gr1.5")
ACS_downscaled_ssp126 = get_datasets("ACS_DS", empat = "ssp126" , timescale=timescale, var='sfcWindmax', grid=("AUS-15", "AUS-10i"))#, "AUS-18")) #get_datasets("CORDEX-CMIP6_Regridded", scenario = "ssp370" , timescale='mon', grid="gr1.5")

In [ ]:
ACS_downscaled_historical

In [ ]:
def preprocess_ACS(ds):
    return ds[target_var].sel(lat=slice(domain_dict['indonesia']['lat_min'], domain_dict['indonesia']['lat_max']), lon=slice(domain_dict['indonesia']['lon_min'], domain_dict['indonesia']['lon_max']))


for target_var in ['pr', 'rsds', 'sfcWindmax', 'tas', 'tasmin', 'tasmax']:
    os.makedirs(f'/g/data/qz03/wp3/data/proj/{timescale}/{target_var}', exist_ok=True)
    for ens_idx in range(0,29):    
        if not any(all(sub in filename for sub in ['ssp370',
                                               str(ACS_downscaled_historical[ens_idx]).split('\n')[-4].split()[-1],
                                               str(ACS_downscaled_historical[ens_idx]).split('\n')[-5].split()[-1],
                                               str(ACS_downscaled_historical[ens_idx]).split('\n')[-7].split()[-1]]) for filename in os.listdir(f'/g/data/qz03/wp3/data/bmkg/proj/{timescale}/{target_var}/')):
            
            print(ACS_downscaled_historical[ens_idx])
            historical_file_list = [file for file in ACS_downscaled_historical[ens_idx].select(var=target_var).get_files() if f'/{target_var}/v' in file]
            ssp126_file_list = [file for file in ACS_downscaled_ssp126[ens_idx].select(var=target_var).get_files() if f'/{target_var}/v' in file]
            ssp370_file_list = [file for file in ACS_downscaled_ssp370[ens_idx].select(var=target_var).get_files() if f'/{target_var}/v' in file]
    
            historical_xr = xr.open_mfdataset(historical_file_list, combine='nested', concat_dim='time', parallel = True, preprocess=preprocess_ACS, engine='netcdf4').compute()
            ssp126_xr = xr.open_mfdataset(ssp126_file_list, combine='nested', concat_dim='time', parallel = True, preprocess=preprocess_ACS, engine='netcdf4').compute()
            ssp370_xr = xr.open_mfdataset(ssp370_file_list, combine='nested', concat_dim='time', parallel = True, preprocess=preprocess_ACS, engine='netcdf4').compute()

            historical_xr.to_netcdf(f"/g/data/qz03/wp3/data/bmkg/proj/{timescale}/{target_var}/{historical_file_list[0].split('/')[-1][:-17]}.nc",\
                                encoding={f'{target_var}': {'zlib': True, 'complevel': 5, 'dtype':'float32'}})
            ssp126_xr.to_netcdf(f"/g/data/qz03/wp3/data/bmkg/proj/{timescale}/{target_var}/{ssp126_file_list[0].split('/')[-1][:-17]}.nc",\
                                encoding={f'{target_var}': {'zlib': True, 'complevel': 5, 'dtype':'float32'}})
            ssp370_xr.to_netcdf(f"/g/data/qz03/wp3/data/bmkg/proj/{timescale}/{target_var}/{ssp370_file_list[0].split('/')[-1][:-17]}.nc",\
                                encoding={f'{target_var}': {'zlib': True, 'complevel': 5, 'dtype':'float32'}})
    
            print()